# 01 · Pipeline basics: three ranges

A GEMM kernel usually has three stages: prologue, mainloop and epilogue.


## 1. The question

`fp16_gemm_0` has prefetch portion of the mainloop split out and is placed after the prologue and before the actual loop:

1. **Prologue** — allocate smem/tmem, initialise mbarriers
2. **Mainloop**  
2.1. **Prefetch** — fill the pipeline stages  
2.2. **k-loop** — iterate over K, interleaving TMA loads and MMA issues  
3. **Epilogue** — drain the accumulator, quantise, TMA-store

**The goal:** here we will add annotations to capture the three major stages using iket ranges.

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        595.58.03
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


### A note on this kernel

This notebook profiles `fp16_gemm_0x`, a **modified** variant of the upstream `fp16_gemm_0`. Upstream hides prefetch inside `cutlass.range(..., prefetch_stages=N)`. The `0x` variant hoists that into an explicit prefetch loop so the phase becomes observable — and, in notebook 02, sweepable. However, the main reason the iket currently does not work inside `cutlass.range()` when `prefetch_stages` is specified.

`kernels/PROVENANCE.md` records exactly what changed. This is worth stating plainly: **instrumenting sometimes requires restructuring**, and you should know when you have done it.

## 2. The instrumentation

Three ranges, one per phase. Nothing else:

```python
iket = cute.experimental.iket

token = iket.range_start("prologue")
# TMA pipeline setup, mbarrier init
iket.range_end(token)

iket.range_push("mainloop")
# k-tile loop: TMA + MMA
iket.range_pop()

iket.range_push("epilogue")
# accumulator drain and TMA store
iket.range_pop()
```

The kernel we profile goes one level deeper, adding per-iteration ranges inside the mainloop. Confirm it changes nothing but instrumentation:

In [3]:
nbhelp.show_instrumentation_diff(
    nbhelp.kernel("modified", "fp16_gemm_0x.py"),
    nbhelp.kernel("instrumented", "fp16_gemm_0x_iket_ps_2_detailed.py"),
    limit=40,
)

added 47 lines, removed 10
NOTE: 8 added line(s) are not IKET calls -- this variant changes more than instrumentation.
------------------------------------------------------------------------
--- fp16_gemm_0x.py
+++ fp16_gemm_0x_iket_ps_2_detailed.py
@@ -83,6 +83,7 @@
     tma_b: cpasync.TmaInfo,
     mC_mnl: cute.Tensor,
 ):
+    cute.experimental.iket.range_push("prologue")
     # Extract tma_tensor from TmaInfo
     mA_mkl = tma_a.tma_tensor
     mB_nkl = tma_b.tma_tensor
@@ -223,18 +224,27 @@
     tCrAcc = cute.make_rmem_tensor(tCgC[None, None, 0].shape, acc_dtype)
     # (TmemCpy,NumTmemCpy)
     tCrC = cute.make_rmem_tensor(tCgC[None, None, 0].shape, io_dtype)
+    cute.experimental.iket.range_pop()
 
     #
     # 2. Main loop
     #
+    cute.experimental.iket.range_push("mainloop")
     num_k_tiles = cute.size(gA, mode=[2])
-    prefetch_stages=ab_stages - 2
+    prefetch_stages = 2
     if warp_idx == 0:
         # Wait for a empty accumulator buffer
         acc_empty = acc_

## 3. Run it

In [4]:
OUT = "out/nb01_ps2"
cmd = (
    f"run-iket --output-dir {OUT} --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel("instrumented", "fp16_gemm_0x_iket_ps_2_detailed.py")} --mnk 4096,4096,512"
)
check_env.run_or_show(cmd, env, est="about 40 seconds")

$ run-iket --output-dir out/nb01_ps2 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============
Running Blackwell fp16 GEMM example 0 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

a: tensor([[ 1., -2., -2.,  ..., -2., -1.,  1.],
        [ 0.,  1., -1.,  ...,  0., -2.,  0.],
        [ 0., -1.,  1.,  ..., -1.,  0.,  0.],
        ...,
        [ 0., -2.,  0.,  ...,  1., -2.,  0.],
        [-1., -1., -1.,  ..., -1.,  0.,  1.],
        [ 1.,  0., -2.,  ...,  0., -1.,  0.]], device='cuda:0',
       dtype=torch.float16)
b: tensor([[-2., -2.,  0.,  ...,  1., -1., -2.],
        [ 1., -2., -1.,  ...,  0., -2.,  0.],
        [-1., -1., -2.,  ...,  0., -1., -1.],
        ...,
        [ 1.,  0.,  0.,  ..., -1., -1., -1.],
        [-2., -2.,  1.,  ...,  0.,  1., -2.],
        [ 1.,  1.,  0.,  ...,  1.,  0., -1.]], device='cuda:0',
       dtype=torch.float1

0

## 4. The trace

In [4]:
nbhelp.figure("nb01/gemm0x_three_ranges_timeline.png", "Three top-level ranges across the kernel: prologue, mainloop, epilogue.")

In [5]:
nbhelp.figure("nb01/gemm0x_three_ranges_detail.png", "The same trace, zoomed to the phase boundaries.")

Going one level deeper inside the mainloop — five events per K-tile — shows what the single `mainloop` band was hiding:

In [6]:
nbhelp.figure("nb01/gemm0x_detailed_mainloop_code.png", "Instrumentation for five events per K-tile.")

In [7]:
nbhelp.figure("nb01/gemm0x_detailed_mainloop_trace.png", "The resulting per-iteration trace.")

In [5]:
try:
    trace = ts.load(ts.find_trace("out/nb01_ps2"))
    ts.print_summary(ts.summarize(trace), top=12)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK11110000_PermutationMNK____MMAAtom_ThrID10_ShapeMNK12825616_TVLayoutA1128161281128_TVLayoutB1256162561256_TVLayoutC11282561281128_cutlasscutenv_0   grid 32x16x1  block 128x1x1  warps traced 2048
warp-lifetime span: 26.1 us

range                      count      median         p90       total
--------------------------------------------------------------------
epilogue                    2048      5.184u      5.376u    9285.60u
acc_wait                    2048      3.040u      3.136u    4790.72u
mainloop                    2048      0.032u      2.880u    1497.25u
k_tile_iter                 4096      0.288u      0.480u    1230.53u
epi_subtile                 8192      0.096u      0.128u     879.07u
acc_release                 2048      0.416u      0.416u     824.10u
prologue                    2048      0.352u      0.384u     736.77u
cons_wait                   4096      0.032u      0.256u     336.83u
prod_acquire                4096      0.03

## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512` — 2,048 warps, 26.2 µs span:

| Range | Median | Count | Total across warps |
|---|---|---|---|
| **`epilogue`** | **5.248 µs** | 2,048 | 9,349 µs |
| `acc_wait` | 3.072 µs | 2,048 | — |
| `prologue` | 0.352 µs | 2,048 | 740 µs |
| `k_tile_iter` | 0.288 µs | 4,096 | 1,236 µs |
| `mainloop` | 0.032 µs | 2,048 | 1,508 µs |


**The epilogue is the largest phase, not the mainloop.** Its 5.2 µs median is an order of magnitude above any single mainloop iteration, and its total across warps (9,349 µs) is six times the mainloop's (1,508 µs).

The reason the mainloop looks small: its per-tile work *overlaps across pipeline stages*, so no single `k_tile_iter` is long. The epilogue has no such overlap — it drains the accumulator, converts, and stores, largely serially.

If you were about to optimise this kernel by tuning the K-loop, the trace just told you to look at the drain instead.

**What would falsify this:** a shape with much larger K. Then the mainloop runs many more iterations against a fixed-cost epilogue and the balance inverts — which is exactly the effect notebook 03 turns into a decision.

**Next:** [02 · Prefetch stages](02_prefetch_stages.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*